In [ ]:
import os
import gc
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import log_loss

import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
plt.style.use('seaborn-v0_8-whitegrid')

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from tqdm.notebook import tqdm

import warnings
warnings.filterwarnings("ignore")

class CFG:
    COMP_PATH = "/kaggle/input/llm-classification-finetuning"
    MODEL_PATH = "/kaggle/input/fb3-deberta-v3-base-baseline-train" 
    
    MAX_LENGTH = 512
    
    BATCH_SIZE = 8
    EPOCHS = 2
    LEARNING_RATE = 1.5e-5
    
    SEED = 42
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    NUM_LABELS = 3

# Fungsi untuk seeding agar hasil reproducible
def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CFG.SEED)
print(f"Menggunakan device: {CFG.DEVICE}")

In [ ]:
train = pd.read_csv(os.path.join(CFG.COMP_PATH, "train.csv"))
test = pd.read_csv(os.path.join(CFG.COMP_PATH, "test.csv"))
submission = pd.read_csv(os.path.join(CFG.COMP_PATH, "sample_submission.csv"))

print("\nDataset berhasil dimuat!")

In [ ]:
print("Data Latih (train):")
print(f"Bentuk: {train.shape}")
train.head()

In [ ]:
print("\nData Uji (test):")
print(f"Bentuk: {test.shape}")
test.head()

In [ ]:
print("\nContoh File Submission:")
print(f"Bentuk: {submission.shape}")
submission.head()

In [ ]:
print("\nStatistik deskriptif data:")
train.describe().T

In [ ]:
print("\nNilai yang hilang :")
train.isnull().sum()

In [ ]:
# Distribusi Target (Winner)
winner_counts = train[['winner_model_a', 'winner_model_b', 'winner_tie']].sum()

plt.figure(figsize=(10, 6))
sns.barplot(x=winner_counts.index, y=winner_counts.values, palette='viridis')
plt.title('Distribusi Pemenang (Model A vs Model B vs Tie)')
plt.ylabel('Jumlah')
plt.xlabel('Hasil')
plt.xticks([0, 1, 2], ['Model A Menang', 'Model B Menang', 'Seri'])
for i, count in enumerate(winner_counts.values):
    plt.text(i, count + 100, str(count), ha='center', fontweight='bold')
plt.show()

In [ ]:
# Analisis Panjang Teks
train['prompt_len'] = train['prompt'].str.len()
train['response_a_len'] = train['response_a'].str.len()
train['response_b_len'] = train['response_b'].str.len()

plt.figure(figsize=(18, 5))

plt.subplot(1, 3, 1)
sns.histplot(train['prompt_len'], bins=50, kde=True, color='blue')
plt.title('Distribusi Panjang Prompt')
plt.xlabel('Jumlah Karakter')

plt.subplot(1, 3, 2)
sns.histplot(train['response_a_len'], bins=50, kde=True, color='green')
plt.title('Distribusi Panjang Respons A')
plt.xlabel('Jumlah Karakter')

plt.subplot(1, 3, 3)
sns.histplot(train['response_b_len'], bins=50, kde=True, color='red')
plt.title('Distribusi Panjang Respons B')
plt.xlabel('Jumlah Karakter')

plt.tight_layout()
plt.show()

In [ ]:
# Frekuensi Model yang Diadu
model_counts = pd.concat([train['model_a'], train['model_b']]).value_counts().head(15)

plt.figure(figsize=(12, 8))
sns.barplot(y=model_counts.index, x=model_counts.values, palette='plasma', orient='h')
plt.title('Top 15 Model yang Paling Sering Diadu', fontsize=16)
plt.xlabel('Frekuensi', fontsize=12)
plt.ylabel('Nama Model', fontsize=12)
plt.show()

In [ ]:
# Word Cloud untuk Prompt
prompt_text = ' '.join(train['prompt'].dropna())
wordcloud = WordCloud(width=1000, height=500, background_color='black', colormap='cividis').generate(prompt_text)

plt.figure(figsize=(15, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud untuk Kata-kata Umum dalam Prompt', fontsize=16)
plt.show()
del prompt_text, wordcloud
gc.collect()

In [ ]:
# Hubungan antara Panjang Respons dan Kemenangan (Verbosity Bias)
train['len_diff'] = train['response_a_len'] - train['response_b_len']
train['winner_label'] = np.select(
    [train['winner_model_a']==1, train['winner_model_b']==1, train['winner_tie']==1],
    ['model_a', 'model_b', 'tie']
)

plt.figure(figsize=(12, 7))
sns.boxplot(x='winner_label', y='len_diff', data=train, palette='coolwarm')
plt.title('Perbedaan Panjang Respons (A - B) vs Pemenang', fontsize=16)
plt.xlabel('Pemenang', fontsize=12)
plt.ylabel('Perbedaan Jumlah Kata (Respons A - Respons B)', fontsize=12)
plt.axhline(0, color='black', linestyle='--')
plt.show()

In [ ]:
# Fungsi untuk membuat input teks tunggal
def create_text_input(df):
    # Menggunakan .fillna('') untuk menangani nilai NaN jika ada
    return "PROMPT: " + df["prompt"].fillna('') + \
           " [SEP] RESPONSE A: " + df["response_a"].fillna('') + \
           " [SEP] RESPONSE B: " + df["response_b"].fillna('')

# Terapkan fungsi ke data training dan test
train['text'] = create_text_input(train)
test['text'] = create_text_input(test)

# Buat kolom target tunggal (0: A, 1: B, 2: Tie)
train['target'] = np.select(
    [train['winner_model_a']==1, train['winner_model_b']==1, train['winner_tie']==1],
    [0, 1, 2]
)

# Inisialisasi tokenizer
tokenizer = AutoTokenizer.from_pretrained(CFG.MODEL_PATH) 

# Bagi data menjadi training dan validasi
train_data, val_data = train_test_split(
    train,
    test_size=0.1,
    random_state=CFG.SEED,
    stratify=train['target']
)

print(f"Ukuran data training: {len(train_data)}")
print(f"Ukuran data validasi: {len(val_data)}")

In [ ]:
# Membuat PyTorch Dataset Class
class PreferenceDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, is_test=False):
        self.df = df
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.texts = df['text'].values
        self.is_test = is_test
        if not self.is_test:
            self.labels = df['target'].values

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        text = self.texts[idx]
        inputs = self.tokenizer(
            text,
            truncation=True,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            return_tensors='pt'
        )
        item = {k: v.squeeze(0) for k, v in inputs.items()}
        if not self.is_test:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

# Mendefinisikan model
model = AutoModelForSequenceClassification.from_pretrained(
    CFG.MODEL_PATH,  
    num_labels=CFG.NUM_LABELS
)
model.to(CFG.DEVICE)
print(f"Model berhasil dimuat dari path lokal: {CFG.MODEL_PATH}")

In [ ]:
# Membuat dataset dan dataloader
train_dataset = PreferenceDataset(train_data, tokenizer, CFG.MAX_LENGTH)
val_dataset = PreferenceDataset(val_data, tokenizer, CFG.MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=CFG.BATCH_SIZE * 2, shuffle=False, num_workers=2)

# Optimizer & Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.LEARNING_RATE)
num_training_steps = len(train_loader) * CFG.EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0.1 * num_training_steps,
    num_training_steps=num_training_steps
)

# Loop Training
best_val_logloss = float('inf')
best_model_path = "best_model.pth"

for epoch in range(CFG.EPOCHS):
    print(f"======== Epoch {epoch + 1}/{CFG.EPOCHS} ========")
    model.train()
    total_train_loss = 0
    progress_bar = tqdm(train_loader, desc="Training")
    
    for batch in progress_bar:
        optimizer.zero_grad()
        inputs = {k: v.to(CFG.DEVICE) for k, v in batch.items()}
        outputs = model(**inputs)
        loss = outputs.loss
        total_train_loss += loss.item()
        loss.backward()
        optimizer.step()
        scheduler.step()
        progress_bar.set_postfix({'loss': loss.item()})
        
    avg_train_loss = total_train_loss / len(train_loader)
    print(f"Average Training Loss: {avg_train_loss:.4f}")

    # Validasi
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validation"):
            labels = batch.pop('labels')
            inputs = {k: v.to(CFG.DEVICE) for k, v in batch.items()}
            outputs = model(**inputs)
            all_preds.append(outputs.logits.cpu().numpy())
            all_labels.append(labels.numpy())
            
    preds_np = np.concatenate(all_preds)
    labels_np = np.concatenate(all_labels)
    probs_np = torch.softmax(torch.tensor(preds_np), dim=1).numpy()
    
    val_logloss = log_loss(labels_np, probs_np)
    print(f"Validation LogLoss: {val_logloss:.4f}")
    
    if val_logloss < best_val_logloss:
        best_val_logloss = val_logloss
        torch.save(model.state_dict(), best_model_path)
        print(f"Model terbaik disimpan di '{best_model_path}' dengan LogLoss: {best_val_logloss:.4f}")
    
    gc.collect()

In [ ]:
model.load_state_dict(torch.load(best_model_path))
model.to(CFG.DEVICE)

print(f"\nSkor LogLoss terbaik pada set validasi adalah: {best_val_logloss:.4f}")

test_dataset = PreferenceDataset(test, tokenizer, CFG.MAX_LENGTH, is_test=True)
test_loader = DataLoader(test_dataset, batch_size=CFG.BATCH_SIZE * 2, shuffle=False, num_workers=2)

def predict(model, loader):
    model.eval()
    all_preds = []
    with torch.no_grad():
        for batch in tqdm(loader, desc="Predicting on Test Data"):
            inputs = {k: v.to(CFG.DEVICE) for k, v in batch.items()}
            outputs = model(**inputs)
            probs = torch.softmax(outputs.logits, dim=1)
            all_preds.append(probs.cpu().numpy())
    return np.concatenate(all_preds)

predictions = predict(model, test_loader)

print("\nPrediksi selesai.")
print("Bentuk array prediksi:", predictions.shape)
print("Contoh 5 probabilitas prediksi pertama:")
print(predictions[:5])

In [ ]:
# Isi dataframe submission dengan hasil prediksi
submission['winner_model_a'] = predictions[:, 0]
submission['winner_model_b'] = predictions[:, 1]
submission['winner_tie'] = predictions[:, 2]

# Simpan ke file csv
submission.to_csv("submission.csv", index=False)

print("\nFile submission.csv berhasil dibuat!")
display(submission.head())